In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import cv2
import os
import numpy as np
from tqdm import tqdm

def extract_masked_regions(input_folder, output_folder, model_path):
    """
    Extract RGB regions from original image where binary mask is 1,
    and set all other pixels to zero (black).
    
    Output structure:
    sam_op_WI/
    ├── image1/
    │   ├── original.jpg
    │   ├── combined_overlay.jpg
    │   ├── mask_000.png
    │   ├── extracted_000.jpg    # RGB content where mask=1, others black
    │   ├── mask_001.png
    │   ├── extracted_001.jpg
    │   ├── ...
    │   └── metadata.txt
    ├── image2/
    └── ...
    """
    # Initialize SAM
    print("Initializing SAM model...")
    sam = sam_model_registry["vit_h"](checkpoint=model_path)
    sam.to(device="cuda")
    mask_generator = SamAutomaticMaskGenerator(sam)
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files
    image_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    print(f"Found {len(image_files)} images to process")
    
    # Process each image
    for img_file in tqdm(image_files, desc="Processing images"):
        try:
            # Read image
            img_path = os.path.join(input_folder, img_file)
            image = cv2.imread(img_path)
            if image is None:
                print(f"Could not read image: {img_file}")
                continue
            
            # Create output directory for this image
            base_name = os.path.splitext(img_file)[0]
            img_output_dir = os.path.join(output_folder, base_name)
            os.makedirs(img_output_dir, exist_ok=True)
            
            # Save original image
            cv2.imwrite(os.path.join(img_output_dir, "original.jpg"), image)
            
            # Convert to RGB for SAM
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Generate masks
            masks = mask_generator.generate(image_rgb)
            
            # Create combined visualization
            combined_overlay = image.copy()
            
            # Process each mask
            for i, mask_data in enumerate(masks):
                mask = mask_data["segmentation"]
                
                # Save binary mask
                mask_binary = (mask * 255).astype(np.uint8)
                cv2.imwrite(os.path.join(img_output_dir, f"mask_{i:03d}.png"), mask_binary)
                
                # Extract RGB content where mask=1, set others to black
                extracted_region = np.zeros_like(image)  # Create black canvas
                extracted_region[mask] = image[mask]     # Copy RGB content where mask=1
                
                # Save extracted region
                cv2.imwrite(os.path.join(img_output_dir, f"extracted_{i:03d}.jpg"), extracted_region)
                
                # Create overlay for visualization (optional)
                overlay = image.copy()
                color = generate_consistent_color(i)
                overlay[mask] = cv2.addWeighted(overlay[mask], 0.6, 
                                              np.full_like(overlay[mask], color), 0.4, 0)
                
                # Add contour
                contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(overlay, contours, -1, color, 2)
                
                # Save overlay (optional)
                cv2.imwrite(os.path.join(img_output_dir, f"overlay_{i:03d}.jpg"), overlay)
                
                # Add to combined visualization
                combined_overlay[mask] = cv2.addWeighted(combined_overlay[mask], 0.7, 
                                                       np.full_like(combined_overlay[mask], color), 0.3, 0)
                cv2.drawContours(combined_overlay, contours, -1, color, 2)
            
            # Save combined overlay
            cv2.imwrite(os.path.join(img_output_dir, "combined_overlay.jpg"), combined_overlay)
            
            # Save metadata
            save_metadata(masks, img_file, img_output_dir)
            
            print(f"Processed {img_file}: {len(masks)} regions extracted")
            
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
    
    print(f"Processing complete! Extracted regions saved to: {output_folder}")

def generate_consistent_color(index):
    """Generate a consistent color based on index"""
    colors = [
        (255, 0, 0),    # Red
        (0, 255, 0),    # Green
        (0, 0, 255),    # Blue
        (255, 255, 0),  # Yellow
        (255, 0, 255),  # Magenta
        (0, 255, 255),  # Cyan
        (255, 128, 0),  # Orange
        (128, 0, 255),  # Purple
        (0, 128, 255),  # Light Blue
        (255, 0, 128),  # Pink
    ]
    return colors[index % len(colors)]

def save_metadata(masks, img_file, output_dir):
    """Save metadata about the masks"""
    with open(os.path.join(output_dir, "metadata.txt"), "w") as f:
        f.write(f"Image: {img_file}\n")
        f.write(f"Number of extracted regions: {len(masks)}\n\n")
        f.write("Region Details:\n")
        f.write("Index | Area (px) | BBox [x,y,w,h] | Predicted IoU | Stability Score\n")
        f.write("-" * 80 + "\n")
        
        for i, mask_data in enumerate(masks):
            f.write(f"{i:5d} | {mask_data['area']:8d} | "
                   f"[{mask_data['bbox'][0]:3d}, {mask_data['bbox'][1]:3d}, "
                   f"{mask_data['bbox'][2]:3d}, {mask_data['bbox'][3]:3d}] | "
                   f"{mask_data['predicted_iou']:11.3f} | {mask_data['stability_score']:14.3f}\n")

# Alternative version with only essential outputs (no overlays)
def extract_masked_regions_only(input_folder, output_folder, model_path):
    """
    Extract only the masked regions without creating overlays
    """
    # Initialize SAM
    print("Initializing SAM model...")
    sam = sam_model_registry["vit_h"](checkpoint=model_path)
    sam.to(device="cuda")
    mask_generator = SamAutomaticMaskGenerator(sam)
    
    os.makedirs(output_folder, exist_ok=True)
    
    image_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    print(f"Found {len(image_files)} images to process")
    
    for img_file in tqdm(image_files, desc="Extracting regions"):
        try:
            img_path = os.path.join(input_folder, img_file)
            image = cv2.imread(img_path)
            if image is None:
                continue
            
            base_name = os.path.splitext(img_file)[0]
            img_output_dir = os.path.join(output_folder, base_name)
            os.makedirs(img_output_dir, exist_ok=True)
            
            # Save original
            cv2.imwrite(os.path.join(img_output_dir, "original.jpg"), image)
            
            # Convert for SAM
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Generate masks
            masks = mask_generator.generate(image_rgb)
            
            # Process each mask
            for i, mask_data in enumerate(masks):
                mask = mask_data["segmentation"]
                
                # Save binary mask
                mask_binary = (mask * 255).astype(np.uint8)
                cv2.imwrite(os.path.join(img_output_dir, f"mask_{i:03d}.png"), mask_binary)
                
                # Extract RGB content where mask=1, set others to black
                extracted_region = np.zeros_like(image)
                extracted_region[mask] = image[mask]
                
                # Save extracted region
                cv2.imwrite(os.path.join(img_output_dir, f"extracted_{i:03d}.jpg"), extracted_region)
            
            # Save simple metadata
            with open(os.path.join(img_output_dir, "metadata.txt"), "w") as f:
                f.write(f"Image: {img_file}\n")
                f.write(f"Number of regions: {len(masks)}\n")
                for i, mask_data in enumerate(masks):
                    f.write(f"Region {i}: area={mask_data['area']} pixels\n")
            
        except Exception as e:
            print(f"Error with {img_file}: {str(e)}")
    
    print(f"Extraction complete! Regions saved to: {output_folder}")

# Configuration
MODEL_PATH = "/home/iiitdmk-param/Desktop/AA/3.1/Codes/sam_vit_h_4b8939.pth"
INPUT_FOLDER = "/home/iiitdmk-param/Desktop/AA/3.1/Data/W_I"
OUTPUT_FOLDER = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2"

# Run the processing (choose one version)
if __name__ == "__main__":
    # Full version with overlays
    extract_masked_regions(INPUT_FOLDER, OUTPUT_FOLDER, MODEL_PATH)
    
    # Or minimal version without overlays
    # extract_masked_regions_only(INPUT_FOLDER, OUTPUT_FOLDER, MODEL_PATH)

In [ ]:
import os

base_dir = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2"

# Collect all file paths (including subfolders)
all_files = []
for root, dirs, files in os.walk(base_dir):
    for file in files:
        all_files.append(os.path.join(root, file))

# Sort the list
all_files = sorted(all_files)

# Print result


print("\n✅ Total files found:", len(all_files))


In [ ]:
import os

base_dir = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2"

all_files = []
mask_files=[]
extracted_files=[]
for root, dirs, files in os.walk(base_dir):
    for file in files:
        all_files.append(os.path.join(root, file))

# Separate lists
mask_files = [f for f in all_files if "mask_" in os.path.basename(f)]
extracted_files = [f for f in all_files if "extracted_" in os.path.basename(f)]

# Sort for consistency
mask_files = sorted(mask_files)
extracted_files = sorted(extracted_files)

print("✅ Mask Files:", len(mask_files))
for f in mask_files:
    print(f)

print("\n✅ Extracted Files:", len(extracted_files))
for f in extracted_files:
    print(f)


In [ ]:
import os
import shutil

src_dir = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2"
dst_dir = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2_extracted_patches"

os.makedirs(dst_dir, exist_ok=True)

for root, dirs, files in os.walk(src_dir):
    for file in files:
        if file.startswith("extracted") and file.endswith(".jpg"):
            folder_name = os.path.basename(root)  # e.g., "airplane40"
            file_num = file.split("_")[-1]        # e.g., "000.jpg"
            new_name = f"{folder_name}_{file_num}"  # airplane40_000.jpg
            
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dst_dir, new_name)
            
            shutil.copy(src_path, dst_path)  # or use os.rename if you want to move instead of copy

print("✅ Renaming and copying done. Saved in:", dst_dir)


In [ ]:
import os
import shutil

src_dir = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2"
dst_dir = "/home/iiitdmk-param/Desktop/AA/3.1/Data/sam_op_WI_2_mask_patches"

os.makedirs(dst_dir, exist_ok=True)

for root, dirs, files in os.walk(src_dir):
    for file in files:
        if file.startswith("mask") and file.endswith(".png"):
            folder_name = os.path.basename(root)  # e.g., "airplane40"
            file_num = file.split("_")[-1]        # e.g., "000.jpg"
            new_name = f"{folder_name}_{file_num}"  # airplane40_000.jpg
            
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dst_dir, new_name)
            
            shutil.copy(src_path, dst_path)  # or use os.rename if you want to move instead of copy

print("✅ Renaming and copying done. Saved in:", dst_dir)